# Final Assignment: Analyzing Historical Stock/Revenue Data and Building a Dashboard

In [11]:
pip install yfinance beautifulsoup4 lxml plotly pandas requests -q

Note: you may need to restart the kernel to use updated packages.


In [12]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [13]:
def make_graph(stock_data, revenue_data, stock):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=("Historical Share Price", "Historical Revenue"),
                        vertical_spacing=.3)
    stock_data_specific = stock_data[pd.to_datetime(stock_data.Date) <= pd.to_datetime("2021-06-14")]
    revenue_data_specific = revenue_data[pd.to_datetime(revenue_data.Date) <= pd.to_datetime("2021-04-30")]
    fig.add_trace(go.Scatter(x=pd.to_datetime(stock_data_specific.Date),
                             y=stock_data_specific.Close.astype("float"), name="Share Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=pd.to_datetime(revenue_data_specific.Date),
                             y=revenue_data_specific.Revenue.astype("float"), name="Revenue"), row=2, col=1)
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(showlegend=False, height=900, title=stock, xaxis_rangeslider_visible=True)
    fig.show(renderer="browser")

## Question 1: Tesla stock data (yfinance)

In [14]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data["Date"] = pd.to_datetime(tesla_data["Date"]).dt.tz_localize(None)
tesla_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


## Question 2: Tesla revenue (web scraping)

In [15]:
def scrape_revenue(url):
    soup = BeautifulSoup(requests.get(url).text, "html.parser")
    rows = []
    for tr in soup.find_all("tbody")[1].find_all("tr"):
        td = tr.find_all("td")
        rows.append({"Date": td[0].text, "Revenue": td[1].text.replace("$", "").replace(",", "")})
    df = pd.DataFrame(rows)
    df = df[df["Revenue"] != ""]
    df.dropna(inplace=True)
    return df

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
tesla_revenue = scrape_revenue(url)
tesla_revenue.tail()

,Date,Revenue
48,2010-09-30,31
49,2010-06-30,28
50,2010-03-31,21
52,2009-09-30,46
53,2009-06-30,27


## Question 3: GameStop stock data (yfinance)

In [16]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data["Date"] = pd.to_datetime(gme_data["Date"]).dt.tz_localize(None)
gme_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13,1.620129,1.693350,1.603296,1.691667,76216000,0.0,0.0
1,2002-02-14,1.712707,1.716074,1.670626,1.683250,11021600,0.0,0.0
2,2002-02-15,1.683250,1.687458,1.658001,1.674834,8389600,0.0,0.0
3,2002-02-19,1.666418,1.666418,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20,1.615920,1.662210,1.603296,1.662210,6892800,0.0,0.0


## Question 4: GameStop revenue (web scraping)

In [17]:
url2 = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
gme_revenue = scrape_revenue(url2)
gme_revenue.tail()

,Date,Revenue
57,2006-01-31,1667
58,2005-10-31,534
59,2005-07-31,416
60,2005-04-30,475
61,2005-01-31,709


## Question 5: Tesla dashboard

In [18]:
make_graph(tesla_data, tesla_revenue, "Tesla")

## Question 6: GameStop dashboard

In [19]:
make_graph(gme_data, gme_revenue, "GameStop")